In [1]:
import datetime
print(datetime.datetime.now())

2026-08-24 17:46:29.810489


## Text input

https://platform.openai.com/docs/models

In [2]:
from dotenv import load_dotenv

load_dotenv()

True

In [4]:
from langchain_openrouter import ChatOpenRouter
model = ChatOpenRouter(model="openai/gpt-5-nano")   # or anthropic/claude-3.5-haiku, etc.

In [5]:
from langchain.agents import create_agent

agent = create_agent(
    model=model,
    system_prompt="You are a science fiction writer, create a capital city at the users request.",
    )

In [6]:
# from langchain.agents import create_agent

# agent = create_agent(
#     model='gpt-5-nano',
#     system_prompt="You are a science fiction writer, create a capital city at the users request.",
# )

In [7]:
from langchain.messages import HumanMessage

question = HumanMessage(content=[
    {"type": "text", "text": "What is the capital of The Moon?"}
])

response = agent.invoke(
    {"messages": [question]}
)

print(response['messages'][-1].content)

Failed to multipart ingest runs: langsmith.utils.LangSmithError: Failed to POST https://eu.api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://eu.api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


In a science-fiction setting where the Moon is a sovereign polity, the capital could be:

Lunaris Prime — the Lunar Commonwealth’s political heart.

A quick picture:
- Location: A domed, sunlit city built inside a stabilized crater on the near side, at the edge of a broad Mare Serenitatis-like basin. Solar towers ring the rim, and light channels (sunstreams) feed the interior terraces.
- Government: The Presidential Palace and Parliament sit in the central Plaza of Arcs, with the Lunar Archives and the Council of Five nearby. It’s the seat of executive, legislative, and judicial power.
- Architecture: White basalt, glass, and lattice metals, layered in terraces and arcades. Everywhere you go, you see skylights, hover-paths, and generous green pockets in shielded courtyards.
- Landmarks: The Lumen Spire (a vertical beacon of data and governance), the Great Lens (a vast solar collector that also doubles as a public observatory), and the Crater Dome District (the residential heart with ma

Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://eu.api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://eu.api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


## Image input

In [8]:
from ipywidgets import FileUpload
from IPython.display import display

uploader = FileUpload(accept='.png', multiple=False)
display(uploader)

FileUpload(value=(), accept='.png', description='Upload')

In [10]:
print(uploader.value)

({'name': 'moon.png', 'type': 'image/png', 'size': 358916, 'content': <memory at 0x000001E191CA2980>, 'last_modified': datetime.datetime(2026, 8, 23, 14, 50, 0, 7000, tzinfo=datetime.timezone.utc)},)


In [11]:
import base64

# Get the first (and only) uploaded file dict
uploaded_file = uploader.value[0]

# This is a memoryview
content_mv = uploaded_file["content"]

# Convert memoryview -> bytes
img_bytes = bytes(content_mv)  # or content_mv.tobytes()

# Now base64 encode
img_b64 = base64.b64encode(img_bytes).decode("utf-8")

In [12]:
multimodal_question = HumanMessage(content=[
    {"type": "text", "text": "Tell me about this capital"},
    {"type": "image", "base64": img_b64, "mime_type": "image/png"}
])

response = agent.invoke(
    {"messages": [multimodal_question]}
)

print(response['messages'][-1].content)

Here’s a portrait of the capital your image suggests. I’m calling it Lumenis Prime, the shining heart of the crescent-hearted world it sits on.

- Where it sits
  - Lumenis Prime rests in a crescent valley ringed by jagged obsidian peaks. The ground is a mix of rust-red soil and glassy plateaus that catch the light and throw it back in prismatic glints. A broad, blue-tinged river threads through the lower terraces, feeding crystal gardens and feeding the city’s energy grid.
  - Above the skyline, a large blue moon or distant planet dominates the sky, giving the city an otherworldly, dusk-to-dawn feel at all hours. The air carries a faint mineral tang from the surrounding rock and the faint hum of large-scale energy collectors.

- Government and society
  - The city is governed by the Synod of Nine, a technocratic-ecclesial council that blends expertise, ethics, and public memory. Each seat presides over a sector: energy, defense, culture, science, resource management, diplomacy, memory

## Audio input

In [16]:
import sounddevice as sd
from scipy.io.wavfile import write
import base64
import io
import time
from tqdm import tqdm

# Recording settings
duration = 5  # seconds
sample_rate = 44100

print("Recording...")
audio = sd.rec(int(duration * sample_rate), samplerate=sample_rate, channels=1)
# Progress bar for the duration
for _ in tqdm(range(duration * 10)):   # update 10× per second
    time.sleep(0.1)
sd.wait()
print("Done.")

# Write WAV to an in-memory buffer
buf = io.BytesIO()
write(buf, sample_rate, audio)
wav_bytes = buf.getvalue()

aud_b64 = base64.b64encode(wav_bytes).decode("utf-8")

Recording...


100%|██████████| 50/50 [00:05<00:00,  9.87it/s]

Done.


In [17]:
agent = create_agent(
    model=model,
)

multimodal_question = HumanMessage(content=[
    {"type": "text", "text": "Tell me about this audio file"},
    {"type": "audio", "base64": aud_b64, "mime_type": "audio/wav"}
])

response = agent.invoke(
    {"messages": [multimodal_question]}
)

print(response['messages'][-1].content)

NotFoundResponseError: No endpoints found that support input audio

Failed to send compressed multipart ingest: Connection error caused failure to POST https://eu.api.smith.langchain.com/runs/multipart in LangSmith API. Please confirm your LANGCHAIN_ENDPOINT. ConnectionError(MaxRetryError("HTTPSConnectionPool(host='eu.api.smith.langchain.com', port=443): Max retries exceeded with url: /runs/multipart (Caused by ProtocolError('Connection aborted.', TimeoutError('The write operation timed out')))"))
Content-Length: 1186269
API Key: lsv2_********************************************d8trace=01a0340d-954a-7fd1-b104-b4dc45758330,id=01a0340d-954a-7fd1-b104-b4dc45758330; trace=01a0340d-954a-7fd1-b104-b4dc45758330,id=01a0340d-954f-7a41-8922-062f06f20b53; trace=01a0340d-954a-7fd1-b104-b4dc45758330,id=01a0340d-9551-76c3-8961-9224f62569dd
Failed to send compressed multipart ingest: Connection error caused failure to POST https://eu.api.smith.langchain.com/runs/multipart in LangSmith API. Please confirm your LANGCHAIN_ENDPOINT. ConnectionError(MaxRetryError("HTTPSCo